[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/bonelab_inverse_dynamics/blob/main/notebooks/tutorial_brick_2d.ipynb)

# One brick carried by a hand: the whole inverse-dynamics workflow on a problem with an exact answer

Tutorial 1 of 4 on the way to the [ten-step quickstart](quickstart_report.ipynb).

You can't see a force. You *can* see where things are, frame after frame, and
Newton says that's enough: know the mass and how the position is changing, and
you know the net force. Inverse dynamics is just that bookkeeping, done
carefully, for a body made of several pieces.

Here it's one piece. A hand we never see carries a brick around (lifts it,
swings it, tilts it), and all we record are four markers on its corners. The
motion is scripted, so I know the hand's force and torque exactly, and I can
grade every step.

We are going to demonstrate the following key steps in the inverse dynamics workflow:

1. **Inputs**: four corner markers, no measured external load, and a skeleton from a scale and a ruler (Steps 1–4).
2. **Markers to pose, pose to derivatives**: an SVD rigid-body fit per frame, then central differences for accelerations (Step 5).
3. **Newton–Euler for one segment**: the hand's force and torque come out as the residual at the top of the chain, from one `inverse_dynamics` call and again by hand (Step 7).
4. **Grading against the exact answer**: our hand wrench on top of the scripted one (Step 8).
5. **The energy audit**: hand power vs the rate of change of the brick's energy (Step 9).
6. **Looking at it**: the brick and its markers in 3-D (Step 10).

And here's what it shows (numbers from the saved outputs below):

1. The rigid fit recovers the pose to round-off: misfit max 2.7e-16 m, rotation error 2.7e-15.
2. Differencing twice costs a little: COM acceleration error 1.19e-05 m/s² against a 0.72 m/s² peak (500 Hz, no noise).
3. The pipeline and the by-hand equations agree exactly (0.0e+00 N and 0.0e+00 N m).
4. The hand wrench matches the analytic one: force error 2.97e-05 N (peak 20.8 N), torque error 5.54e-06 N m (peak 1.47 N m).
5. Energy balances: peak imbalance 8.17e-06 W of a 7.55 W peak.

Steps are numbered the way the quickstart numbers them. A few are trivial here
(no floor, no contact, no second segment), but I kept them, one line each, so
the *shape* of the workflow is already the final one.

In [1]:
# Colab bootstrap — a no-op when running locally in the uv environment.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "boneid @ git+https://github.com/jeremydwong/bonelab_inverse_dynamics"],
                   check=True)
os.makedirs("../reports", exist_ok=True)

## Study config (the only thing you edit)

The brick got weighed and measured with a ruler. For a rigid body that's the
whole "anthropometrics" step: mass, dimensions, and where the hand holds it
(the centre of the top face, which I call the segment's *proximal end*, the
end nearer whatever is moving it).

In [2]:
import numpy as np

BRICK_MASS_KG = 2.0
BRICK_SIZE_M = np.array([0.15, 0.10, 0.30])   # x thickness, y width, z length (long axis)
RATE_HZ = 500.0                                 # mocap rate
DURATION_S = 2.0
KINEMATIC_LOWPASS_HZ = 0.0                      # analytic data: filtering would only add error

## Step 1: inputs, part 1 — four corner markers

`simulate_brick_2d` scripts the hand's motion and gives back what a camera
system would: marker trajectories, `[T, 3]` metres each, lab frame, z up. The
four corners are picked so they do **not** lie in one plane. Three
non-collinear points already fix a rigid pose; a fourth off that plane keeps
the fit well conditioned in every orientation.

We also keep the simulator's `truth` (the exact hand wrench), but only to
grade ourselves. Nothing reads it before Step 8.

In [3]:
from boneid.simulate import simulate_brick_2d, tetrahedral_corners

skeleton_true, kin_true, markers, truth = simulate_brick_2d(
    rate=RATE_HZ, duration=DURATION_S, mass=BRICK_MASS_KG, size=BRICK_SIZE_M)
t = kin_true.t
print(f"{len(t)} frames at {RATE_HZ:.0f} Hz, {len(markers)} markers: {list(markers)}")
for name, xyz in markers.items():
    print(f"  {name}: starts at {np.round(xyz[0], 3)} m")

1000 frames at 500 Hz, 4 markers: ['c1', 'c2', 'c3', 'c4']
  c1: starts at [0.075 0.05  0.75 ] m
  c2: starts at [ 0.075 -0.05   0.45 ] m
  c3: starts at [-0.075  0.05   0.45 ] m
  c4: starts at [-0.075 -0.05   0.75 ] m


In [4]:
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")
from IPython.display import HTML, display
from boneid import report as R
from boneid.report import JOINT_COLORS, FAINT, NEUTRAL, new_fig, new_grid, style_axes

def show(fig):
    # the report's own SVG path: identical output to the HTML reports
    display(HTML(f'<img src="{R.fig_svg(fig)}" style="max-width:100%">'))

fig, ax = new_fig(height=3.4, width=6.0)
for k, (name, xyz) in enumerate(markers.items()):
    ax.plot(xyz[:, 0], xyz[:, 2], color=JOINT_COLORS[k], lw=1.4, label=name)
    ax.plot(xyz[0, 0], xyz[0, 2], "o", color=JOINT_COLORS[k], ms=5)
ax.set_aspect("equal")
style_axes(ax, "x, forward (m)", "z, up (m)")
ax.legend(frameon=False, fontsize=9)
ax.set_title("marker paths in the sagittal plane (dots = start)", color=NEUTRAL, fontsize=10)
show(fig)

## Step 2: inputs, part 2 — force data (none measured)

In a gait trial the force plate measures the load at the *bottom* of the
chain and the unknown sits at the top. Here nothing touches the brick but the
hand, so the measured external load is exactly zero and the hand's wrench is
the unknown at the top. I say so explicitly with a zero `GroundWrench`.

(A *wrench* is a force plus a moment about a stated point. boneid carries
every external load this way rather than as a force at a centre of pressure;
[two bricks](tutorial_two_bricks_2d.ipynb) explains why.)

In [5]:
from boneid.core import GroundWrench

nothing = GroundWrench(t=t, rate=RATE_HZ, force=np.zeros((len(t), 3)),
                       moment=np.zeros((len(t), 3)), point=np.zeros(3))
print("external load on the brick's far end:", nothing.force.max(), "N")

external load on the brick's far end: 0.0 N


## Step 3: inputs, part 3 — skeleton (mass, centre of mass, inertia)

A `Skeleton` is the list of segment parameters the physics needs: mass, the
centre of mass (COM) in the segment's own frame, and the inertia tensor about
that COM. For a homogeneous box all three come from the scale and the ruler.

The segment frame's origin is the grip; the COM is half a length below it
along local z. For a uniform box the inertia about the COM is
$I = \tfrac{m}{12}\,\mathrm{diag}(y^2+z^2,\; x^2+z^2,\; x^2+y^2)$, which is
one line, `box_inertia`.

In [6]:
from boneid.core import Skeleton
from boneid.simulate import box_inertia

L = BRICK_SIZE_M[2]
skeleton = Skeleton(segment_names=["brick"], joint_names=[],
                    mass=np.array([BRICK_MASS_KG]),
                    com_local=np.array([[0.0, 0.0, -0.5 * L]]),
                    inertia_local=box_inertia(BRICK_MASS_KG, BRICK_SIZE_M)[None],
                    length=np.array([L]))
print("inertia about the COM (kg m^2):")
print(np.round(skeleton.inertia_local[0], 5))

inertia about the COM (kg m^2):
[[0.01667 0.      0.     ]
 [0.      0.01875 0.     ]
 [0.      0.      0.00542]]


## Step 4: inputs, part 4 — analysis parameters

Gravity, the contact threshold and the low-pass cutoffs live in one record.
The data is analytic and noise-free, so both filters are off. On real data
this is the cell whose values matter most (more on that at the end).

In [7]:
from boneid.core import AnalysisParams

params = AnalysisParams(lowpass_hz=KINEMATIC_LOWPASS_HZ, force_lowpass_hz=0.0)
params

AnalysisParams(gravity=array([ 0.  ,  0.  , -9.81]), contact_threshold_n=20.0, lowpass_hz=0.0, force_lowpass_hz=0.0, filter_order=4, treadmill_speed=0.0, flag_crossover=False)

## Step 5: markers to pose, pose to derivatives

This is where dots become a body. We know where the four corners sit **in
the brick's own frame** (the ruler again, `tetrahedral_corners`), and we see
where they are **in the lab** every frame. The rigid transform carrying one
onto the other is the pose: a rotation $R(t)$ and translation $d(t)$ with

$$\text{marker}_k(t) \approx R(t)\,\text{corner}_k + d(t).$$

`fit_rigid_transform` solves this in the least-squares sense with the
Söderkvist & Wedin (1993) SVD method, the same one the legacy MATLAB used
(`SODER`) and the running study's cluster fits use. It also returns the RMS
misfit per frame. On real skin markers that's the first data-quality number I
look at; here it's round-off.

The corner coordinates were measured relative to the grip, so $d(t)$ *is*
the grip's trajectory, and the far end is one length down local z.

In [8]:
from boneid.core import SegmentKinematics, fit_rigid_transform

corners = tetrahedral_corners(BRICK_SIZE_M)                      # [4,3] in the brick frame
moving = np.stack([markers[f"c{k + 1}"] for k in range(4)], axis=1)  # [T,4,3] in the lab
r, d, misfit = fit_rigid_transform(corners, moving)

prox = d                                                          # the grip
dist = d + np.einsum("tij,j->ti", r, [0.0, 0.0, -L])              # the far end
kin = SegmentKinematics(t=t, rate=RATE_HZ, r_world=r[:, None],
                        prox_pos=prox[:, None], dist_pos=dist[:, None])
print(f"rigid-fit misfit: max {misfit.max():.1e} m  (real skin markers: millimetres)")
print(f"pose error vs the scripted motion: rotation {np.abs(r - kin_true.r_world[:, 0]).max():.1e}, "
      f"position {np.abs(prox - kin_true.prox_pos[:, 0]).max():.1e} m")

rigid-fit misfit: max 2.7e-16 m  (real skin markers: millimetres)
pose error vs the scripted motion: rotation 2.7e-15, position 2.2e-16 m


### Pose to derivatives: what the physics actually consumes

Newton needs the COM acceleration; Euler needs angular velocity and
acceleration. All come from the pose by central differences
(`chain_kinematics`, which `inverse_dynamics` calls internally). Two
derivatives of position is where noise gets amplified. The plot puts our
differenced accelerations on the analytic ones; with no noise at all, the gap
is just finite-difference truncation (printed below the plot).

In [9]:
from boneid.core import chain_kinematics

ck = chain_kinematics(skeleton, kin, lowpass_hz=params.lowpass_hz)
fig, axes = new_grid(1, 2, height=2.9)
axes[0, 0].plot(t, truth["a_com"][:, 2], color=NEUTRAL, lw=2.6, alpha=0.4, label="analytic")
axes[0, 0].plot(t, ck["a_com"][:, 0, 2], color=JOINT_COLORS[0], lw=1.1, label="central difference")
style_axes(axes[0, 0], "time (s)", "COM acceleration, z (m/s²)")
axes[0, 0].legend(frameon=False, fontsize=8.5)
axes[0, 1].plot(t, truth["alpha_y"], color=NEUTRAL, lw=2.6, alpha=0.4, label="analytic")
axes[0, 1].plot(t, ck["alpha"][:, 0, 1], color=JOINT_COLORS[1], lw=1.1, label="central difference")
style_axes(axes[0, 1], "time (s)", "angular acceleration, y (rad/s²)")
axes[0, 1].legend(frameon=False, fontsize=8.5)
show(fig)
inner = slice(5, -5)
print(f"acceleration error {np.abs(ck['a_com'][inner, 0, 2] - truth['a_com'][inner, 2]).max():.2e} m/s²  "
      f"of peak {np.abs(truth['a_com']).max():.2f}")

acceleration error 1.19e-05 m/s²  of peak 0.72


## Step 6: contact events (nothing to do)

No floor, nothing to detect. On walking data this step finds when each foot
is on its belt; for the brick it's a no-op. (The next tutorial has a floor.)

## Step 7: Newton–Euler for one segment — one call, then by hand

`inverse_dynamics` walks the chain from the far end toward the top. For each
segment it balances forces and moments:

$$F_p = m\,a_{com} - F_d - m\,g, \qquad
M_p = I_w\,\alpha + \omega\times(I_w\,\omega) - M_d - (p_d - c)\times F_d - (p_p - c)\times F_p$$

where $(F_d, M_d)$ is the load on the far end (zero here), $(F_p, M_p)$ the
load at the near end, $c$ the COM, and $I_w = R\,I\,R^{\mathsf T}$ the inertia
in the lab frame. With one segment there are no joints, so the near-end load
is reported as the **residual**: the wrench that has to act at the top of the
chain for the balance to close. For the brick, the residual *is* the hand.

Below, the same two equations typed out by hand. They agree exactly.

In [10]:
from boneid.core import inverse_dynamics, inertia_world

idres = inverse_dynamics(skeleton, kin, nothing, params)
hand_force, hand_torque = idres.residual_force, idres.residual_torque

# — by hand, straight from the two equations —
g = params.gravity
m = skeleton.mass[0]
com, a, w, al = ck["com"][:, 0], ck["a_com"][:, 0], ck["omega"][:, 0], ck["alpha"][:, 0]
I_w = inertia_world(ck["r"][:, 0], skeleton.inertia_local[0])
F = m * a - m * g
M = (np.einsum("tij,tj->ti", I_w, al) + np.cross(w, np.einsum("tij,tj->ti", I_w, w))
     - np.cross(prox - com, F))
print(f"pipeline vs by hand: force {np.abs(F - hand_force).max():.1e} N, "
      f"torque {np.abs(M - hand_torque).max():.1e} N m")

pipeline vs by hand: force 0.0e+00 N, torque 0.0e+00 N m


## Step 8: grading against the exact answer — the hand's force and torque

Grey is the analytic wrench the simulator used; ours sits on top. Vertical
force starts and ends at the brick's weight (19.6 N) and rises while the
brick is being accelerated upward; the torque is what it takes to swing a
0.3 m box about a grip at one end.

In [11]:
fig, axes = new_grid(1, 2, height=3.0)
for c, lab in ((0, "x, forward"), (2, "z, up")):
    axes[0, 0].plot(t, truth["force"][:, c], color=NEUTRAL, lw=2.6, alpha=0.4)
    axes[0, 0].plot(t, hand_force[:, c], color=JOINT_COLORS[c // 2], lw=1.2, label=lab)
axes[0, 0].axhline(m * 9.81, color=FAINT, lw=1, ls="--")
axes[0, 0].annotate("brick weight", (t[-1], m * 9.81), ha="right", va="bottom", fontsize=8.5, color=NEUTRAL)
style_axes(axes[0, 0], "time (s)", "hand force (N)")
axes[0, 0].legend(frameon=False, fontsize=8.5)
axes[0, 1].plot(t, truth["torque"][:, 1], color=NEUTRAL, lw=2.6, alpha=0.4, label="analytic")
axes[0, 1].plot(t, hand_torque[:, 1], color=JOINT_COLORS[1], lw=1.2, label="ours")
style_axes(axes[0, 1], "time (s)", "hand torque about the grip, y (N m)")
axes[0, 1].legend(frameon=False, fontsize=8.5)
show(fig)
print(f"force error  {np.abs(hand_force[inner] - truth['force'][inner]).max():.2e} N   "
      f"(peak {np.abs(truth['force']).max():.1f} N)")
print(f"torque error {np.abs(hand_torque[inner] - truth['torque'][inner]).max():.2e} N m "
      f"(peak {np.abs(truth['torque']).max():.2f} N m)")

force error  2.97e-05 N   (peak 20.8 N)
torque error 5.54e-06 N m (peak 1.47 N m)


## Step 9: the energy audit

An independent check: the hand's power, $F\cdot v_{grip} + M\cdot\omega$,
has to equal the rate of change of the brick's kinetic plus potential energy.
Energy is never used inside the recursion, so when the two curves coincide
it's evidence, not bookkeeping. Every boneid run ends with this audit.

In [12]:
from boneid.core import energy_audit

audit = energy_audit(skeleton, kin, nothing, idres, params)
fig, axes = new_grid(1, 2, height=2.9)
axes[0, 0].plot(t, audit.de_dt, color=NEUTRAL, lw=2.6, alpha=0.45, label="d(KE+PE)/dt")
axes[0, 0].plot(t, audit.power_total, color=JOINT_COLORS[0], lw=1.2, label="hand power")
style_axes(axes[0, 0], "time (s)", "power (W)")
axes[0, 0].legend(frameon=False, fontsize=8.5)
axes[0, 1].plot(t[inner], audit.imbalance[inner], color=JOINT_COLORS[1], lw=1.2)
style_axes(axes[0, 1], "time (s)", "imbalance (W)")
show(fig)
print(f"peak imbalance {np.abs(audit.imbalance[inner]).max():.2e} W of "
      f"{np.abs(audit.de_dt).max():.2f} W peak")

peak imbalance 8.17e-06 W of 7.55 W peak


## Step 10: looking at it in 3-D

The brick, its four markers (red) and the path. Drag to orbit; the clip loops.

In [13]:
from html import escape
from pathlib import Path
from boneid.report import simulated_viewer_html
from boneid.viz import box_mesh

viewer_html = simulated_viewer_html(skeleton, kin, markers=markers,
                                    segment_meshes={0: box_mesh(BRICK_SIZE_M)},
                                    decimate=10, camera_offset=(1.2, -1.6, 0.3))
Path("../reports/tutorial_brick_2d_viewer.html").write_text(viewer_html)
display(HTML(f'<iframe srcdoc="{escape(viewer_html, quote=True)}" width="100%" height="420"></iframe>'))

## What to carry to the next page

- **Markers → pose → derivatives → forces.** Every inverse-dynamics pipeline
  is this chain. The only inputs were marker positions, a mass, and a ruler.
- **The residual is the top-of-chain load.** With nothing above the brick it
  *is* the answer. When a chain stands on the ground and its top is free, the
  same number should be zero, and how far it is from zero is most of the
  diagnostic story in the later reports.
- **Two derivatives amplify noise.** With perfect data the acceleration error
  was 1.19e-05 m/s² on a 0.72 m/s² peak, truncation alone. Add a millimetre of
  marker noise at 500 Hz and I think noise swamps that completely; that's what
  Step 4's cutoff is for, and why it's a study decision rather than a default.

Next: [two bricks](tutorial_two_bricks_2d.ipynb), a joint between them, a
floor under them, and the first residual that *should* be zero.